# Chapter 7 — Working with Text Data

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain why text data needs special preprocessing.
2. Convert text into numerical features using Bag-of-Words.
3. Use TF-IDF to reweight text features.
4. Understand how n-grams capture context.
5. Apply simple topic modeling with NMF.

## Chapter Overview

Text data is unstructured. Machine learning models cannot directly process raw sentences, so text must be converted into numerical representations. A common classical approach is Bag-of-Words, where each document is represented by word counts.

This notebook uses a small Indonesian sentiment-like dataset for lightweight demonstration, while the theoretical concepts follow the chapter's text processing workflow.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Small Text Dataset

The dataset contains simple positive and negative movie-review-like sentences.

### Important note

In real projects, text datasets are usually much larger. This small dataset is used only to demonstrate the mechanics of vectorization and classification.

In [2]:
texts = [
    'film ini sangat bagus dan ceritanya menarik',
    'akting pemain sangat memukau dan alurnya bagus',
    'saya suka film ini karena visualnya indah',
    'cerita bagus musik bagus dan akhir yang memuaskan',
    'film ini buruk dan sangat membosankan',
    'alur cerita jelek akting buruk dan tidak menarik',
    'saya tidak suka film ini karena terlalu lambat',
    'visual buruk cerita membingungkan dan akhir mengecewakan'
]
labels = np.array([1, 1, 1, 1, 0, 0, 0, 0])

for text, label in zip(texts, labels):
    print(label, '-', text)

1 - film ini sangat bagus dan ceritanya menarik
1 - akting pemain sangat memukau dan alurnya bagus
1 - saya suka film ini karena visualnya indah
1 - cerita bagus musik bagus dan akhir yang memuaskan
0 - film ini buruk dan sangat membosankan
0 - alur cerita jelek akting buruk dan tidak menarik
0 - saya tidak suka film ini karena terlalu lambat
0 - visual buruk cerita membingungkan dan akhir mengecewakan


## 2. Bag-of-Words with CountVectorizer

Bag-of-Words represents each document by counting word occurrences.

### Theory

The model ignores word order in the basic unigram version. This is simple and effective, but it can lose context. For example, the words `not` and `good` may be counted separately even though `not good` has a specific meaning.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(texts)

print('Bag-of-Words matrix shape:', X_bow.shape)
print('Vocabulary:')
print(vectorizer.get_feature_names_out())
print('\nDense representation:')
print(pd.DataFrame(X_bow.toarray(), columns=vectorizer.get_feature_names_out()))

Bag-of-Words matrix shape: (8, 31)
Vocabulary:
['akhir' 'akting' 'alur' 'alurnya' 'bagus' 'buruk' 'cerita' 'ceritanya'
 'dan' 'film' 'indah' 'ini' 'jelek' 'karena' 'lambat' 'membingungkan'
 'membosankan' 'memuaskan' 'memukau' 'menarik' 'mengecewakan' 'musik'
 'pemain' 'sangat' 'saya' 'suka' 'terlalu' 'tidak' 'visual' 'visualnya'
 'yang']

Dense representation:
   akhir  akting  alur  alurnya  bagus  buruk  cerita  ceritanya  dan  film  \
0      0       0     0        0      1      0       0          1    1     1   
1      0       1     0        1      1      0       0          0    1     0   
2      0       0     0        0      0      0       0          0    0     1   
3      1       0     0        0      2      0       1          0    1     0   
4      0       0     0        0      0      1       0          0    1     1   
5      0       1     1        0      0      1       1          0    1     0   
6      0       0     0        0      0      0       0          0    0     1   
7    

### Output Interpretation

Rows represent documents and columns represent vocabulary terms. Each value indicates how many times a word appears in a document. The matrix is usually sparse because most documents contain only a small subset of all vocabulary words.

## 3. Text Classification

Once text has been vectorized, standard machine learning models can be applied.

This example uses Logistic Regression because it is a strong baseline for high-dimensional sparse text features.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

text_model = make_pipeline(
    CountVectorizer(),
    LogisticRegression()
)

text_model.fit(texts, labels)

new_reviews = [
    'film bagus dan menarik',
    'film buruk dan membosankan',
    'akting bagus tetapi cerita lambat'
]

predictions = text_model.predict(new_reviews)
for review, pred in zip(new_reviews, predictions):
    label = 'positive' if pred == 1 else 'negative'
    print(f'{review!r} -> {label}')

'film bagus dan menarik' -> positive
'film buruk dan membosankan' -> negative
'akting bagus tetapi cerita lambat' -> positive


### Output Interpretation

The model predicts sentiment based on words learned from the training examples. Because the dataset is very small, predictions should not be considered reliable for real applications. The goal is to demonstrate the workflow.

## 4. TF-IDF and n-Grams

### TF-IDF

TF-IDF reduces the weight of very common words and increases the importance of words that are more informative in specific documents.

### n-Grams

n-grams represent sequences of words. Bigrams can capture short phrases such as `tidak suka`, which may be more meaningful than individual words.

In [5]:
tfidf_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression()
)

tfidf_model.fit(texts, labels)
print('TF-IDF predictions:', tfidf_model.predict(new_reviews))

ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_ngram = ngram_vectorizer.fit_transform(texts)

print('Number of unigram + bigram features:', len(ngram_vectorizer.get_feature_names_out()))
print('Example features:')
print(ngram_vectorizer.get_feature_names_out()[:30])

TF-IDF predictions: [1 0 1]
Number of unigram + bigram features: 73
Example features:
['akhir' 'akhir mengecewakan' 'akhir yang' 'akting' 'akting buruk'
 'akting pemain' 'alur' 'alur cerita' 'alurnya' 'alurnya bagus' 'bagus'
 'bagus dan' 'bagus musik' 'buruk' 'buruk cerita' 'buruk dan' 'cerita'
 'cerita bagus' 'cerita jelek' 'cerita membingungkan' 'ceritanya'
 'ceritanya menarik' 'dan' 'dan akhir' 'dan alurnya' 'dan ceritanya'
 'dan sangat' 'dan tidak' 'film' 'film ini']


### Output Interpretation

Using n-grams increases the number of features because the model now considers both individual words and word pairs. This can improve context understanding, but it may also increase overfitting if the dataset is small.

## 5. Topic Modeling with NMF

Topic modeling attempts to discover hidden themes in a collection of documents.

NMF decomposes the document-term matrix into topics and document-topic weights. In this simple example, topics may roughly correspond to positive and negative words.

In [6]:
from sklearn.decomposition import NMF

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(texts)

nmf = NMF(n_components=2, random_state=RANDOM_STATE, init='nndsvda', max_iter=500)
nmf.fit(X_tfidf)

feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(nmf.components_):
    top_indices = topic.argsort()[-5:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    print(f'Topic {topic_idx}:', ', '.join(top_words))

Topic 0: dan, bagus, sangat, buruk, cerita
Topic 1: saya, suka, karena, ini, film


### Output Interpretation

The top words in each topic indicate the dominant terms associated with that topic. With a larger corpus, topic modeling can reveal meaningful document themes. With a tiny dataset, it mainly demonstrates the method.

## Key Takeaways

- Text must be transformed into numerical features before modeling.
- Bag-of-Words is simple and effective but ignores word order.
- TF-IDF reweights terms based on informativeness.
- n-grams capture short context but increase feature dimensionality.
- Topic modeling can help explore hidden themes in text collections.

## Chapter Summary

Chapter 7 introduces classical natural language processing workflows in machine learning. It shows how text becomes structured numerical data through vectorization and how standard models can then be applied.